_Change for each model_

In [1]:
# After reference samples are collected, find their polysemanticity level
# Algorithm:
#   1) get dict: layer: neuron: refs
#   2) create concept database: tensor(n_refs, hidden_dim)
#   3) compute polysemanticity score

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import json
import torch

from torchvision.models import ResNet50_Weights

from  configs.general_cofig import *
from interpretability.visualizations.visualization_functions import report_polysem, plot_polysem, visualize_neuron
from models.create_model import create_resnet50
from datasets.imagenet_mini import ImagenetMini

from semanticlens.component_visualization.activation_based import CustomActivationComponentVisualizer
from semanticlens.component_visualization.aggregators import aggregate_conv_mean
from semanticlens.foundation_models import ClipMobile, SigLipV2
from semanticlens.utils.helper import get_polysem_score

weights = ResNet50_Weights.IMAGENET1K_V2
transform = weights.transforms()

model = create_resnet50()
model_name = "resnet_50"
model_name_cache = "resnet_50"

layer = ["layer1.0.conv2", "layer1.1.conv2", "layer1.2.conv2",
         "layer2.0.conv2", "layer2.1.conv2", "layer2.2.conv2", "layer2.3.conv2",
         "layer3.0.conv2", "layer3.1.conv2", "layer3.2.conv2", "layer3.3.conv2", "layer3.4.conv2", "layer3.5.conv2",
         "layer4.0.conv2", "layer4.1.conv2", "layer4.2.conv2"]

aggr_func = aggregate_conv_mean

NUM_SAMPLES = 10


Using device: cpu


_Model independent_

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model.eval().to(device)
model.name = model_name

if EMBEDDING_MODEL == "CLIP_Mobile":
    fm = ClipMobile(device=device)
elif EMBEDDING_MODEL == "SigLipV2":
    fm = SigLipV2(device=device)
else:
    print("Unknown embedding model")

dataset = ImagenetMini(
    wordnet_ids_to_classes_path=f"{ROOT}/datasets/input_files/wordnet_ids_to_classes.txt"
)
dataset.name = "imagenetmini-1000"

top_k_ref_samples = torch.load(
    f"{OUTPUTS}/top_k_act_neurons/"
    f"top_{K_ACT_NEURONS}_max_samples_{model_name}.pt",
    map_location="cpu",
    weights_only=False,
)

cv_obj = CustomActivationComponentVisualizer(
    model=model,
    dataset_model=dataset, # TODO: add transform
    dataset_fm=dataset,
    layer_names=layer,
    num_samples=NUM_SAMPLES,
    device="cpu",
    aggregate_fn= aggr_func,
    cache_dir=f"{OUTPUTS}/semanticlens_cache",
    model_name=model_name,
)
cv_obj.run()
concept_db = cv_obj._compute_concept_db(fm, top_k_ref_samples, batch_size=64)
polysem_scores = get_polysem_score(concept_db)


finding top k neurons
Loading top k neurons stats. from existing files 
Crop_db was loaded from an existing file. It contains 5000 crops.
creating crop_db for layer layer1.0.conv2
creating crop_db for layer layer1.1.conv2
creating crop_db for layer layer1.2.conv2
creating crop_db for layer layer2.0.conv2
creating crop_db for layer layer2.1.conv2
creating crop_db for layer layer2.2.conv2
creating crop_db for layer layer2.3.conv2
creating crop_db for layer layer3.0.conv2
creating crop_db for layer layer3.1.conv2
creating crop_db for layer layer3.2.conv2
creating crop_db for layer layer3.3.conv2
creating crop_db for layer layer3.4.conv2
creating crop_db for layer layer3.5.conv2
creating crop_db for layer layer4.0.conv2
creating crop_db for layer layer4.1.conv2
Embedding the dataset


In [ ]:
torch.save(polysem_scores,
           f"{OUTPUTS}/polysemanticity_scores/{EMBEDDING_MODEL}_{NUM_SAMPLES}_ref_samples/polysemanticity_score_{model_name}.pt")